# Hidden Markov Model for Named Entity Recognition

In [1]:
import pandas as pd
import collections
import math
from sklearn.metrics import classification_report

In [2]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}

df_train = pd.read_parquet("hf://datasets/lhoestq/conll2003/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/lhoestq/conll2003/" + splits["test"])

C:\Users\BluRay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_train.head()

,id,tokens,pos_tags,chunk_tags,ner_tags
0,0,"[EU, rejects, German, call, to, boycott, Briti...","[22, 42, 16, 21, 35, 37, 16, 21, 7]","[11, 21, 11, 12, 21, 22, 11, 12, 0]","[3, 0, 7, 0, 0, 0, 7, 0, 0]"
1,1,"[Peter, Blackburn]","[22, 22]","[11, 12]","[1, 2]"
2,2,"[BRUSSELS, 1996-08-22]","[22, 11]","[11, 12]","[5, 0]"
3,3,"[The, European, Commission, said, on, Thursday...","[12, 22, 22, 38, 15, 22, 28, 38, 15, 16, 21, 3...","[11, 12, 12, 21, 13, 11, 11, 21, 13, 11, 12, 1...","[0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, ..."
4,4,"[Germany, 's, representative, to, the, Europea...","[22, 27, 21, 35, 12, 22, 22, 27, 16, 21, 22, 2...","[11, 11, 12, 13, 11, 12, 12, 11, 12, 12, 12, 1...","[5, 0, 0, 0, 0, 3, 4, 0, 0, 0, 1, 2, 0, 0, 0, ..."


Since we only need NER tags, we can ignore POS and chunk tags.

In [4]:
df_train.drop(columns=["chunk_tags", "pos_tags"], inplace=True)
df_test.drop(columns=["chunk_tags", "pos_tags"], inplace=True)

df_train.head()

,id,tokens,ner_tags
0,0,"[EU, rejects, German, call, to, boycott, Briti...","[3, 0, 7, 0, 0, 0, 7, 0, 0]"
1,1,"[Peter, Blackburn]","[1, 2]"
2,2,"[BRUSSELS, 1996-08-22]","[5, 0]"
3,3,"[The, European, Commission, said, on, Thursday...","[0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, ..."
4,4,"[Germany, 's, representative, to, the, Europea...","[5, 0, 0, 0, 0, 3, 4, 0, 0, 0, 1, 2, 0, 0, 0, ..."


#### NER Tags

0: "O" (Outside of a named entity)

1: "B-PER" (Beginning of a person's name)

2: "I-PER" (Inside of a person's name)

3: "B-ORG" (Beginning of an organization)

4: "I-ORG" (Inside of an organization)

5: "B-LOC" (Beginning of a location)

6: "I-LOC" (Inside of a location)

7: "B-MISC" (Beginning of a miscellaneous entity)

8: "I-MISC" (Inside of a miscellaneous entity)

## Model Training

### Counting Tags and Transitions

In [5]:
tag_counts = collections.defaultdict(int)
transition_counts = collections.defaultdict(int)
emission_counts = collections.defaultdict(int)

# each row represents a sentence, which contain tonkens and NER tags
for _, row in df_train.iterrows():
    tokens = row['tokens']
    tags = row['ner_tags']
    
    # starting tag
    prev_tag = "<START>"
    tag_counts[prev_tag] += 1
    
    # iterate through the words and tags at the same time
    for word, tag in zip(tokens, tags):
        # increment counts
        tag_counts[tag] += 1
        transition_counts[(prev_tag, tag)] += 1
        emission_counts[(tag, word)] += 1
        
        # use current tag for next iteration
        prev_tag = tag

### Calculating Probabilities Using Counts

In [6]:
transition_probs = {}
for (prev_tag, tag), count in transition_counts.items():
    transition_probs[(prev_tag, tag)] = count / tag_counts[prev_tag]

emission_probs = {}
for (tag, word), count in emission_counts.items():
    emission_probs[(tag, word)] = count / tag_counts[tag]

In [7]:
for transition in list(transition_probs.items()):
    print(transition)

(('<START>', np.int64(3)), 0.17484509650309807)
((np.int64(3), np.int64(0)), 0.586141433317513)
((np.int64(0), np.int64(7)), 0.016871292266685538)
((np.int64(7), np.int64(0)), 0.705933682373473)
((np.int64(0), np.int64(0)), 0.8190095413320124)
(('<START>', np.int64(1)), 0.09607577807848444)
((np.int64(1), np.int64(2)), 0.649090909090909)
(('<START>', np.int64(5)), 0.11259881774802365)
((np.int64(5), np.int64(0)), 0.8323529411764706)
(('<START>', np.int64(0)), 0.5807278683854427)
((np.int64(0), np.int64(3)), 0.022361391218200473)
((np.int64(3), np.int64(4)), 0.39313399778516056)
((np.int64(4), np.int64(0)), 0.6479481641468683)
((np.int64(0), np.int64(1)), 0.030564106193020323)
((np.int64(2), np.int64(0)), 0.9070229681978799)
((np.int64(0), np.int64(5)), 0.03262805316727406)
((np.int64(2), np.int64(2)), 0.053886925795053005)
((np.int64(1), np.int64(0)), 0.34045454545454545)
((np.int64(7), np.int64(8)), 0.24956369982547993)
((np.int64(8), np.int64(8)), 0.2571428571428571)
((np.int64(8), n

In [8]:
print("\nEmissions:")
for emission in list(emission_probs.items()):
    print(emission)


Emissions:
((np.int64(3), 'EU'), 0.0037968675842429997)
((np.int64(0), 'rejects'), 5.896991355010673e-06)
((np.int64(7), 'German'), 0.02443280977312391)
((np.int64(0), 'call'), 0.0001769097406503202)
((np.int64(0), 'to'), 0.0199436247626461)
((np.int64(0), 'boycott'), 2.9484956775053368e-05)
((np.int64(7), 'British'), 0.02268760907504363)
((np.int64(0), 'lamb'), 1.769097406503202e-05)
((np.int64(0), '.'), 0.04341365035558858)
((np.int64(1), 'Peter'), 0.004696969696969697)
((np.int64(2), 'Blackburn'), 0.00022084805653710247)
((np.int64(5), 'BRUSSELS'), 0.002240896358543417)
((np.int64(0), '1996-08-22'), 0.0007371239193763342)
((np.int64(0), 'The'), 0.006445411551026666)
((np.int64(3), 'European'), 0.004587881664293625)
((np.int64(4), 'Commission'), 0.0091792656587473)
((np.int64(0), 'said'), 0.010885846041349703)
((np.int64(0), 'on'), 0.012006274398801732)
((np.int64(0), 'Thursday'), 0.001680642536178042)
((np.int64(0), 'it'), 0.0032610362193209023)
((np.int64(0), 'disagreed'), 1.17939

## Viterbi Decoding

Viterbi function takes as arguments:
- `tokens`: the sentence to be decoded
- `tags`: the NER tags from 0 to 8 (without `<start>`)
- `transition_probabilities`

In [ ]:
def viterbi(tokens, tags, transition_probabilities, emission_probs):
    
    epsilon = 1e-6 # prevent zero probabilities
    
    # stores probabilities and paths for each step
    viterbi_path = []
    backpointers = []
    
    # first token initialization (root node)
    first_token = tokens[0]
    first_viterbi = {}
    first_backpointer = {}
    
    # if the first token is completely unknown (out of vocabulary)
    # true when all tags have zero emission probability for the first token (doesn't appear in training data)
    is_oov = all(emission_probs.get((t, first_token), 0) == 0 for t in tags)
    
    for tag in tags:
        transition_prob = transition_probabilities.get(("<START>", tag), epsilon)
        emission_prob = epsilon if is_oov else emission_probs.get((tag, first_token), epsilon)
        
        # multiply transition and emission probabilities for the first token (log space to prevent underflow)
        first_viterbi[tag] = math.log(transition_prob) + math.log(emission_prob)
        first_backpointer[tag] = "<START>"
    
        # first_viterbi[tag] = trans_prob * emiss_prob
        # first_backpointer[tag] = "<START>"
        
    # root node probabilities
    viterbi_path.append(first_viterbi)
    backpointers.append(first_backpointer)
    
    # forward pass for the whole sequence of tokens
    for t in range(1, len(tokens)):
        token = tokens[t]
        current_viterbi = {}
        current_backpointer = {}
        
        is_oov = all(emission_probs.get((tag, token), 0) == 0 for tag in tags)
        
        # for each token, go through all tags and calculate best score and backpointer
        for current_tag in tags:
            
            best_score = -float('inf') # minimum score
            best_prev_tag = None
            
            # save emission probability for current tag
            emission_prob = epsilon if is_oov else emission_probs.get((current_tag, token), epsilon)
            emission_score = math.log(emission_prob)
            
            # for each possible previous current token, go through all previous tags
            # and calculate the score of transitioning from every previous tag to current one 
            for prev_tag in tags:
                transition_prob = transition_probabilities.get((prev_tag, current_tag), epsilon)
                transition_score = math.log(transition_prob)
                
                # multiply transition and emission probabilities with the previous step's probability (log space)
                # use the previous tags score
                score = viterbi_path[t-1][prev_tag] + transition_score + emission_score
                
                # pick best probability/score and keep track of the best previous tag
                if score > best_score:
                    best_score = score
                    best_prev_tag = prev_tag
          
          # for current token and tag, save the best score and backpointer      
            current_viterbi[current_tag] = best_score
            current_backpointer[current_tag] = best_prev_tag
          
        # for each token after going through all tags and calculating scores, save the best scores and backpointers  
        viterbi_path.append(current_viterbi)
        backpointers.append(current_backpointer)
        
    # forward pass done
        
    # backtracking for best sequence
    best_last_tag = max(viterbi_path[-1], key=viterbi_path[-1].get) # last dictionary, get tag with highest score (key value)
    best_sequence = [best_last_tag]
    
    # get best sequence by following backpointers, start from the end of the sequence
    for t in range(len(tokens) - 1, 0, -1):
        best_prev_tag = backpointers[t][best_last_tag]
        best_sequence.insert(0, best_prev_tag) # insert at the start of the list
        best_last_tag = best_prev_tag
        
    return best_sequence

## Evaluation

In [10]:
true_labels = []
predicted_labels = []

tags = [0, 1, 2, 3, 4, 5, 6, 7, 8]
tag_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

for _, row in df_test.iterrows():
    tokens = row['tokens']
    true_tags = row['ner_tags']
    
    # get predictions for the current sentence
    pred_tags = viterbi(tokens, tags, transition_probs, emission_probs)
    
    # store the actual and predicted tags
    true_labels.extend(true_tags)
    predicted_labels.extend(pred_tags)

# Generate the evaluation report
report = classification_report(true_labels, predicted_labels, target_names=tag_names, zero_division=0)
print(report)

              precision    recall  f1-score   support

           O       0.94      0.99      0.97     38323
       B-PER       0.93      0.48      0.63      1617
       I-PER       0.97      0.58      0.72      1156
       B-ORG       0.83      0.61      0.70      1661
       I-ORG       0.85      0.61      0.71       835
       B-LOC       0.85      0.81      0.83      1668
       I-LOC       0.78      0.66      0.72       257
      B-MISC       0.82      0.73      0.77       702
      I-MISC       0.71      0.64      0.68       216

    accuracy                           0.93     46435
   macro avg       0.85      0.68      0.75     46435
weighted avg       0.93      0.93      0.92     46435

